In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
df = pd.read_csv("../data/processed/healthcare_processed.csv")

raw_df = pd.read_csv("../data/raw/HealthCare.csv")

X = df.drop(columns=["No_show"])
y = df["No_show"]

groups = raw_df["PatientId"]

print("Dataset shape:", X.shape)

Dataset shape: (110527, 17)


In [3]:
#Recreate the patient-level development split

In [4]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Development data:", X_train.shape)
print("Final test data:", X_test.shape)

Development data: (88491, 17)
Final test data: (22036, 17)


In [5]:
# Define the preprocessing

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "Age",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received",
    "WaitingDays",
    "ScheduledHour",
    "AppointmentMonth"
]

categorical_features = [
    "Gender",
    "Neighbourhood",
    "ScheduledDayOfWeek",
    "AppointmentDayOfWeek",
    "AgeGroup",
    "WaitingGroup",
    "ScheduledTimeGroup"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [7]:
# Create the final Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

production_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        max_features="sqrt",
        min_samples_leaf=1,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

In [9]:
# Train on all development data

In [10]:
production_model.fit(
    X_train,
    y_train
)

print("Production model training completed.")

Production model training completed.


In [11]:
# Test that the model works

In [12]:
test_probability = production_model.predict_proba(
    X_test
)[:, 1]

test_prediction = (
    test_probability >= 0.24
).astype(int)

print("Test predictions generated successfully.")
print(test_probability[:10])
print(test_prediction[:10])

Test predictions generated successfully.
[0.01643759 0.15368954 0.31668102 0.28289332 0.24045187 0.24900319
 0.24069878 0.0226248  0.4114116  0.05276681]
[0 0 1 1 1 1 1 0 1 0]


In [13]:
#Save the complete pipeline

In [14]:
joblib.dump(
    production_model,
    "../models/model.pkl"
)

print("Production model saved successfully.")

Production model saved successfully.


In [15]:
# Verify the saved model

In [16]:
loaded_model = joblib.load(
    "../models/model.pkl"
)

print("Model loaded successfully.")
print(type(loaded_model))

Model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


In [17]:
loaded_probability = loaded_model.predict_proba(
    X_test.head(5)
)[:, 1]

print(loaded_probability)

[0.01643759 0.15368954 0.31668102 0.28289332 0.24045187]
